# Aula 11 - PySpark: NOT EXISTS e LEFT ANTI JOIN

In [0]:
# Aula 11 - NOT EXISTS e LEFT ANTI JOIN
# Objetivo: encontrar registros presentes em 'final' mas AUSENTES em 'source'
# (oposto da Aula 10, que buscava a interseção com EXISTS / left_semi)

# Imports: traz col(), regexp_replace, etc. para o namespace
from pyspark.sql.functions import *

# Leitura do CSV com header e encoding UTF-8
# .distinct() remove linhas totalmente duplicadas logo na carga
df_carros = (
    spark.read
    .format('csv')
    .option('header', 'true')
    .option('encoding', 'utf-8')
    .load('/Workspace/Users/jcesarsantana215@gmail.com/Aulas PySpark/datasets/modelo_carro.csv', sep=',')
    .distinct()
)

# Subconjunto 'source': apenas os carros de id 1, 2 e 3 (referência menor)
df_carros_source = df_carros.where(
    (col("id_carro") == '1') |
    (col("id_carro") == '2') |
    (col("id_carro") == '3')
)

# Subconjunto 'final': ids 1, 2, 3 e 4 (tem 1 registro a mais que o source)
# O id 4 será o resultado esperado do NOT EXISTS / leftanti
df_carros_final = df_carros.where(
    (col("id_carro") == '1') |
    (col("id_carro") == '2') |
    (col("id_carro") == '3') |
    (col("id_carro") == '4')
)

# Exibe os dois DataFrames para comparar (final tem 1 linha extra)
display(df_carros_source)

display(df_carros_final)

In [0]:
# Cria views temporárias para permitir consultas SQL nos DataFrames
# 'carros_source' = referência (3 linhas) | 'carros_final' = conjunto maior (4 linhas)
df_carros_source.createOrReplaceTempView("carros_source")
df_carros_final.createOrReplaceTempView("carros_final")

In [0]:
%sql
-- NOT EXISTS via célula %sql (magic do Databricks)
-- Retorna linhas de carros_final cujo id_carro NÃO aparece em carros_source
-- Ou seja: anti-join lógico — esperado: apenas o carro de id 4
SELECT * FROM carros_final AS cf
WHERE NOT EXISTS(
    SELECT 1 FROM carros_source AS cs WHERE cs.id_carro = cf.id_carro
)

In [0]:
# Mesmo NOT EXISTS, agora via spark.sql() (retorna um DataFrame manipulável)
# SELECT 1 na subquery é convenção: basta verificar existência, sem trazer colunas
# Útil quando o resultado precisa seguir no pipeline PySpark (filtros, joins, escrita)
df_spark_sql = spark.sql("""
SELECT * FROM carros_final AS cf
WHERE NOT EXISTS(
    SELECT 1 FROM carros_source AS cs WHERE cs.id_carro = cf.id_carro
)
""")
display(df_spark_sql)

In [0]:
# LEFT ANTI JOIN: equivalente PySpark do NOT EXISTS (sem SQL)
# 'leftanti' mantém só as linhas da esquerda SEM correspondência na direita
# É a operação espelho do 'left_semi' visto na Aula 10
df_carros_spark = df_carros_final.join(
    df_carros_source,
    (df_carros_final.id_carro == df_carros_source.id_carro),
    'leftanti'
)

display(df_carros_spark)